# Bước 0: gộp LoRA vào Qwen3-4B và đo lại dev (HUONG_DAN_GPU "Bước 0"; làm TRƯỚC vì cần cho bài nộp dù A5 có đạt hay không). Khoảng **40 đến 60 phút GPU**.
BTC không cho đưa base model vào Docker, nên bài nộp phải là mô hình đã gộp LoRA. Notebook gộp `q4b_lora` vào Qwen3-4B (`scripts/merge_lora.py`, gộp float32, lưu bfloat16), chạy lại **mốc** `q4b_lora` và đo **model đã gộp (không adapter)** trên dev 155 mẫu, chữ sạch và ASR `phowl_ft`, luật bật (mốc 66,82 và 56,98), so từng cặp, tính sha256, rồi lưu `q4b_merged` và `phowl_ft/merged` vào output để chuyển cho chủ dự án.

**Trước khi chạy cell đầu tiên:** đặt notebook **Private**, Internet **Off**, Accelerator **GPU (RTX PRO 6000)**. Cần khoảng 20 GB RAM CPU khi gộp (cell M-1 in `free -g`).
**Input cần gắn (4 thứ):**
1. dataset `lifemate-code` bản mới từ commit `b2853d2` (có `merge_lora.py`)
2. dataset `train+public_test` (raw)
3. dataset `thuan-qwen3-4b`
4. **output của notebook `Hello_done`** (adapter `q4b_lora`, ASR `phowl_ft/merged`, `state/processed`)

**KHÔNG gắn** output A4, A4b, A5 hay `train-hist` (cùng có `state/`, gây khôi phục nhầm).
**Thứ tự cell:** P-1 đến P-3, M-1 (kiểm tra, sha256 adapter và ASR so với `models.lock.json`), M-2 (mốc `q4b_lora`), M-3 (gộp, sha256), M-4 (đo model đã gộp), M-5 (so từng cặp: **hai số dev cần gửi**), M-6 (đóng zip), M-7 (lưu trạng thái, chạy cuối rồi Save Version, output khoảng 12 GB).
**Gửi Phương:** `merge_results.zip` (điểm, so sánh, sha256) và hai số dev ở M-5. **Chuyển cho chủ dự án** (Google Drive riêng, không công khai) `checkpoints/q4b_merged/*` (khoảng 8 GB), `checkpoints/phowl_ft/merged/*` (khoảng 3 GB) và `SHA256SUMS_merged.txt`: tải từng file ở tab Output; kiểm sau khi tải bằng `sha256sum -c SHA256SUMS_merged.txt`.
**Nếu dev của model đã gộp tụt so với `q4b_lora`** (chênh lớn hơn nhiễu): báo ngay, đừng chuyển trọng số; Phương cần xử lý trước khi nộp.

In [ ]:
%%bash
# P-1: setup. Dùng kaggle_setup.sh của Phương; chỉ đặt đường dẫn model cho khớp đúng thứ ta gắn:
#  - QWEN = Qwen3-4B (hidden 2560, 36 tầng), tìm theo cấu hình vì còn có Qwen3-8B/Qwen3.5-9B trong input (không lấy "qwen3 đầu tiên")
#  - PHO = đường dẫn giả (A5 không dùng PhoWhisper)
SRC=$(find /kaggle/input -name kaggle_setup.sh -print -quit)
[ -n "$SRC" ] || { echo "ERROR: không thấy kaggle_setup.sh, dataset code đã gắn chưa?"; exit 1; }
find /kaggle/input -maxdepth 8 -name config.json 2>/dev/null > /kaggle/working/config_paths.txt
QW=$(python -X utf8 - <<'EOF'
import json
from pathlib import Path
for line in open("/kaggle/working/config_paths.txt", encoding="utf-8"):
    p = Path(line.strip())
    try:
        c = json.loads(p.read_text(encoding="utf-8"))
    except Exception:
        continue
    t = c.get("text_config", c)
    if c.get("model_type") == "qwen3" and t.get("hidden_size") == 2560 and t.get("num_hidden_layers") == 36:
        print(p.parent)
        break
EOF
)
echo "Qwen3-4B dir: ${QW:-KHÔNG THẤY}"
[ -n "$QW" ] || { echo "ERROR: không thấy Qwen3-4B (hidden 2560, 36 tầng). Gắn dataset thuan-qwen3-4b."; exit 1; }
cp "$SRC" /kaggle/working/setup_patched.sh
sed -i -e "s#^QWEN=.*#QWEN=$QW#" -e 's#^PHO=.*#PHO=/nonexistent/phowhisper#' -e 's/python -I /python /g' /kaggle/working/setup_patched.sh
bash /kaggle/working/setup_patched.sh
sed -i 's/python -I /python /g' /kaggle/working/lifemate/scripts/*.sh
echo "--- đã khôi phục từ output notebook trước? (checkpoints và outputs)"; ls /kaggle/working/checkpoints | head -20; ls /kaggle/working/outputs | head -12

In [ ]:
%%bash
# P-2: in phiên bản torch/CUDA/GPU, kiểm tra GPU, peft, accelerate và việc transformers nhận qwen3_5
pip uninstall -y torchao 2>&1 | tail -1
python -X utf8 - <<'EOF' | tee /kaggle/working/torch_cuda.txt
import sys
import torch
print(torch.__version__, torch.version.cuda, sys.version)
import peft, transformers
print("peft", peft.__version__, "| transformers", transformers.__version__, "| gpus", torch.cuda.device_count(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ GPU")
from transformers.models.auto.configuration_auto import CONFIG_MAPPING
print("transformers nhận qwen3:", "qwen3" in CONFIG_MAPPING, "| qwen3_5 (Qwen3.5):", "qwen3_5" in CONFIG_MAPPING)
try:
    import accelerate
    print("accelerate", accelerate.__version__)
except ImportError:
    print("THIẾU accelerate: gen_text.py sẽ lỗi ở bước nạp model (device_map). Báo lại, đừng pip install từ mạng.")
if not torch.cuda.is_available():
    raise SystemExit("Phiên này KHÔNG có GPU: bật Accelerator rồi chạy lại")
print("VRAM tổng: %.0f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))
EOF

In [ ]:
%%bash
# P-3: ghi sẵn đoạn mã đóng gói (chỉ file kết quả nhỏ; không có trọng số)
mkdir -p /kaggle/working/helpers
cat > /kaggle/working/helpers/pack_zip.py <<'PYEOF'
"""Đóng gói file kết quả nhỏ thành một zip kèm MANIFEST sha256 (không đóng gói trọng số).
usage: python pack_zip.py <out.zip> <file> [<file> ...]   (file không tồn tại thì bỏ qua và báo)"""
import hashlib
import sys
import zipfile
from pathlib import Path


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while block := f.read(1 << 20):
            h.update(block)
    return h.hexdigest()


out = Path(sys.argv[1])
files = []
for name in sys.argv[2:]:
    p = Path(name)
    if p.is_file():
        files.append(p)
    else:
        print("bỏ qua (không có):", name)
files = sorted(set(files), key=str)
manifest = "".join(f"{sha256(p)}  {p.as_posix()}  ({p.stat().st_size} bytes)\n" for p in files)
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for p in files:
        z.write(p, p.as_posix().lstrip("/"))
    z.writestr("MANIFEST_sha256.txt", manifest)
print(f"{out}: {out.stat().st_size / 1e6:.2f} MB, {len(files)} file")
PYEOF
ls -la /kaggle/working/helpers

In [ ]:
%%bash
# M-1: kiểm tra đủ đồ cho BƯỚC 0 (gộp LoRA): mã mới, adapter q4b_lora, ASR đã gộp phowl_ft/merged, bản ghi ASR của dev. Thiếu thì DỪNG.
set -uo pipefail
source /kaggle/working/lifemate/kaggle_env.sh
MISSING=0
need() { if [ -s "$1" ]; then echo "ok     $1"; else echo "THIẾU  $1"; MISSING=1; fi; }
need scripts/merge_lora.py
need scripts/compare_runs.py
need scripts/infer_multi_gpu.sh
need scripts/eval.py
need models.lock.json
need checkpoints/q4b_lora/adapter_model.safetensors
need checkpoints/q4b_lora/adapter_config.json
need checkpoints/phowl_ft/merged/config.json
need data/processed/dev_ids.txt
need data/processed/asr_phowl_ft_train.jsonl
need data/processed/gold_hash_train.json
if [ "$MISSING" = 1 ]; then echo "-> Kiểm tra input: lifemate-code (bản từ b2853d2, có merge_lora.py), train+public_test, thuan-qwen3-4b, output notebook Hello_done."; exit 1; fi
n=$(wc -l < data/processed/dev_ids.txt)
[ "$n" = 155 ] || { echo "dev_ids.txt có $n dòng, phải là 155"; exit 1; }
echo "dev_ids.txt: 155 id (đúng)"
ls -la checkpoints/phowl_ft/merged | head -12
echo "-- đối chiếu sha256 với models.lock.json (adapter và ASR)"
python -X utf8 - <<'EOF'
import hashlib, json
from pathlib import Path
lock = json.load(open("models.lock.json", encoding="utf-8"))
def sha(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        while b := f.read(1 << 24):
            h.update(b)
    return h.hexdigest()
for key, path in (("adapter", "checkpoints/q4b_lora"), ("asr", "checkpoints/phowl_ft/merged")):
    for name, want in lock[key]["weights"].items():
        p = Path(path) / name
        if not p.is_file():
            print(f"{key}: THIẾU {p}"); continue
        got = sha(p)
        print(f"{key}: {name} {'TRÙNG models.lock.json' if got == want else 'KHÁC models.lock.json (báo Phương)'} ({got[:12]}… so với {want[:12]}…)")
EOF
echo "-- dọn bản sao không dùng (giữ q4b_lora và phowl_ft/merged)"
for d in checkpoints/*; do case "$(basename "$d")" in q4b_lora|phowl_ft) ;; *) rm -rf "$d" ;; esac; done
rm -rf checkpoints/q4b_lora/checkpoint-*
for d in checkpoints/phowl_ft/*; do [ "$(basename "$d")" = "merged" ] || rm -rf "$d"; done
rm -rf /kaggle/working/outputs/*
mkdir -p outputs/logs
du -sh checkpoints | cut -f1 | xargs echo "checkpoints sau khi dọn:"
df -h /kaggle/working | tail -1
free -g | head -2

In [ ]:
%%bash
# M-2 (khoảng 6 phút): chạy lại MỐC q4b_lora (Qwen3-4B + adapter) trên dev 155 mẫu, chữ sạch và ASR phowl_ft, luật bật, ngay trong phiên này
# (để so với model đã gộp cùng môi trường). Kỳ vọng: 66,82 (sạch) và 56,98 (ASR phowl_ft).
set -euo pipefail
source /kaggle/working/lifemate/kaggle_env.sh
D=data/processed
bash scripts/infer_multi_gpu.sh train outputs/dev_q4b_lora_clean.jsonl --ids $D/dev_ids.txt --backend hf --model $QWEN --adapter checkpoints/q4b_lora --text clean
python -X utf8 scripts/eval.py --sub outputs/dev_q4b_lora_clean.jsonl --ids $D/dev_ids.txt | tee outputs/logs/m0_eval_q4b_lora_clean.txt
bash scripts/infer_multi_gpu.sh train outputs/dev_q4b_lora_phowl_ft.jsonl --ids $D/dev_ids.txt --backend hf --model $QWEN --adapter checkpoints/q4b_lora --text asr:phowl_ft
python -X utf8 scripts/eval.py --sub outputs/dev_q4b_lora_phowl_ft.jsonl --ids $D/dev_ids.txt | tee outputs/logs/m0_eval_q4b_lora_phowl_ft.txt

In [ ]:
%%bash
# M-3 (khoảng 5 đến 10 phút, cần khoảng 20 GB RAM): GỘP LoRA vào Qwen3-4B -> checkpoints/q4b_merged (float32 khi gộp, lưu bfloat16), rồi tính sha256.
# Đúng lệnh của Phương (HUONG_DAN_GPU mục "Bước 0"). BTC không cho đưa base model vào Docker nên bài nộp là model đã gộp.
set -euo pipefail
source /kaggle/working/lifemate/kaggle_env.sh
START=$(date +%s)
python -X utf8 scripts/merge_lora.py --base "$QWEN" --adapter checkpoints/q4b_lora --out checkpoints/q4b_merged 2>&1 | tee outputs/logs/m0_merge.txt | tail -n 5
echo "thời gian gộp: $(( $(date +%s) - START )) s" | tee -a outputs/logs/m0_merge.txt
ls -la checkpoints/q4b_merged
echo "-- sha256 (dạng sha256sum -c) của q4b_merged và phowl_ft/merged, để chuyển cho chủ dự án"
( cd checkpoints && find q4b_merged phowl_ft/merged -type f | sort | xargs sha256sum ) | tee /kaggle/working/SHA256SUMS_merged.txt
du -sh checkpoints/q4b_merged checkpoints/phowl_ft/merged

In [ ]:
%%bash
# M-4 (khoảng 6 phút): đo dev với model ĐÃ GỘP, KHÔNG adapter (chữ sạch và ASR phowl_ft), luật bật, đúng lệnh của Phương. Rồi so với mốc q4b_lora vừa chạy ở M-2.
set -euo pipefail
source /kaggle/working/lifemate/kaggle_env.sh
D=data/processed
bash scripts/infer_multi_gpu.sh train outputs/dev_merged_clean.jsonl --ids $D/dev_ids.txt --backend hf --model checkpoints/q4b_merged --text clean
python -X utf8 scripts/eval.py --sub outputs/dev_merged_clean.jsonl --ids $D/dev_ids.txt | tee outputs/logs/m0_eval_merged_clean.txt
bash scripts/infer_multi_gpu.sh train outputs/dev_merged_phowl_ft.jsonl --ids $D/dev_ids.txt --backend hf --model checkpoints/q4b_merged --text asr:phowl_ft
python -X utf8 scripts/eval.py --sub outputs/dev_merged_phowl_ft.jsonl --ids $D/dev_ids.txt | tee outputs/logs/m0_eval_merged_phowl_ft.txt

In [ ]:
%%bash
# M-5: so từng cặp q4b_lora (Qwen3-4B + adapter) với q4b_merged (đã gộp, không adapter) trên dev: số mẫu khác nhau và chênh điểm. HAI SỐ DEV CẦN GỬI nằm ở đây.
set -uo pipefail
source /kaggle/working/lifemate/kaggle_env.sh
D=data/processed
{
echo "################ chữ sạch: q4b_lora -> q4b_merged   (kỳ vọng mốc 66,82)"
python -X utf8 scripts/compare_runs.py --a outputs/dev_q4b_lora_clean.jsonl --b outputs/dev_merged_clean.jsonl --ids $D/dev_ids.txt --names q4b_lora,q4b_merged
echo
echo "################ ASR phowl_ft: q4b_lora -> q4b_merged   (kỳ vọng mốc 56,98)"
python -X utf8 scripts/compare_runs.py --a outputs/dev_q4b_lora_phowl_ft.jsonl --b outputs/dev_merged_phowl_ft.jsonl --ids $D/dev_ids.txt --names q4b_lora,q4b_merged
echo
echo "################ số dòng giống hệt (tool_calls và state_hash) giữa q4b_lora và q4b_merged"
python -X utf8 - <<'EOF'
import json
for tag in ("clean", "phowl_ft"):
    def load(p):
        return {json.loads(l)["id"]: (json.loads(l)["tool_calls"], json.loads(l)["state_hash"]) for l in open(p, encoding="utf-8") if l.strip()}
    a, b = load(f"outputs/dev_q4b_lora_{tag}.jsonl"), load(f"outputs/dev_merged_{tag}.jsonl")
    diff = sum(1 for i in a if a[i] != b.get(i))
    print(f"{tag:9}: {len(a) - diff}/{len(a)} mẫu giống hệt, {diff} mẫu khác")
EOF
} 2>&1 | tee outputs/logs/m0_compare.txt

In [ ]:
%%bash
# M-6: gói kết quả nhỏ để gửi Phương: merge_results.zip (điểm, so sánh, sha256 của model đã gộp, log, môi trường). KHÔNG có trọng số.
set -uo pipefail
source /kaggle/working/lifemate/kaggle_env.sh
python -X utf8 /kaggle/working/helpers/pack_zip.py /kaggle/working/merge_results.zip \
  outputs/dev_q4b_lora_clean.jsonl outputs/dev_q4b_lora_phowl_ft.jsonl outputs/dev_merged_clean.jsonl outputs/dev_merged_phowl_ft.jsonl \
  outputs/logs/m0_*.txt /kaggle/working/SHA256SUMS_merged.txt /kaggle/working/torch_cuda.txt
ls -la /kaggle/working/merge_results.zip

In [ ]:
%%bash
# M-7 (CUỐI CÙNG): lưu trạng thái rồi Save Version. Giữ checkpoints/q4b_merged, checkpoints/phowl_ft/merged (và q4b_lora nhỏ), state/processed, outputs/, SHA256SUMS_merged.txt, merge_results.zip.
# Cell này xóa thư mục làm việc lifemate/. Output khoảng 12 GB (giới hạn 20 GB).
cd /kaggle/working
rm -rf checkpoints/q4b_lora/checkpoint-*
bash /kaggle/working/lifemate/scripts/kaggle_save_state.sh
rm -rf helpers setup_patched.sh config_paths.txt
du -sh /kaggle/working/state /kaggle/working/checkpoints /kaggle/working/outputs 2>/dev/null
ls /kaggle/working /kaggle/working/checkpoints
echo
echo "TẢI VỀ để chuyển cho chủ dự án (tab Output của notebook): checkpoints/q4b_merged/* (khoảng 8 GB), checkpoints/phowl_ft/merged/* (khoảng 3 GB), SHA256SUMS_merged.txt."
echo "Kaggle cho tải từng file; kiểm sau khi tải: sha256sum -c SHA256SUMS_merged.txt (đặt file trong thư mục chứa q4b_merged/ và phowl_ft/merged/)."